In [ ]:
# ============================================
# SIMPLE DATA ANALYSIS / MODEL PREP CHECK
# ============================================

import pandas as pd
import numpy as np

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from google.colab import files

uploaded = files.upload()
file_name = next(iter(uploaded))

df = pd.read_csv(file_name)

# ---------- 1. Basic overview ----------
print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)

print(f"Rows:    {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")

display(df.head())

print("\nData types:")
display(df.dtypes.to_frame("dtype"))


# ---------- 2. Missing values ----------
print("\n" + "=" * 60)
print("MISSING VALUES")
print("=" * 60)

missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2)
})

missing = missing[missing["missing_count"] > 0] \
    .sort_values("missing_percent", ascending=False)

if len(missing):
    display(missing)
else:
    print("No missing values found!")


# ---------- 3. Duplicate rows ----------
print("\n" + "=" * 60)
print("DUPLICATES")
print("=" * 60)

print(f"Duplicate rows: {df.duplicated().sum():,}")


# ---------- 4. Unique values ----------
print("\n" + "=" * 60)
print("UNIQUE VALUES")
print("=" * 60)

unique = pd.DataFrame({
    "unique_values": df.nunique(),
    "dtype": df.dtypes.astype(str)
}).sort_values("unique_values")

display(unique)


# ---------- 5. Numeric summary ----------
print("\n" + "=" * 60)
print("NUMERIC SUMMARY")
print("=" * 60)

display(df.describe().T)


# ---------- 6. Categorical / text columns ----------
print("\n" + "=" * 60)
print("CATEGORICAL / TEXT COLUMNS")
print("=" * 60)

categorical_cols = df.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print(categorical_cols)

for col in categorical_cols:
    print(f"\n--- {col} ---")
    print(f"Unique values: {df[col].nunique(dropna=True)}")
    display(df[col].value_counts(dropna=False).head(10))


# ---------- 7. Potential ID / high-cardinality columns ----------
print("\n" + "=" * 60)
print("HIGH-CARDINALITY COLUMNS")
print("=" * 60)

high_cardinality = []

for col in df.columns:
    unique_ratio = df[col].nunique(dropna=True) / len(df)
    if unique_ratio > 0.8:
        high_cardinality.append({
            "column": col,
            "unique_values": df[col].nunique(),
            "unique_ratio": round(unique_ratio, 3)
        })

display(pd.DataFrame(high_cardinality))


# ---------- 9. Constant / almost constant columns ----------
print("\n" + "=" * 60)
print("CONSTANT / NEAR-CONSTANT COLUMNS")
print("=" * 60)

constant_cols = [
    col for col in df.columns
    if df[col].nunique(dropna=False) <= 1
]

near_constant_cols = [
    col for col in df.columns
    if df[col].nunique(dropna=True) <= 2
]

print("Constant columns:")
print(constant_cols)

print("\nColumns with <= 2 unique values:")
print(near_constant_cols)


# ---------- 10. Correlations between numeric variables ----------
print("\n" + "=" * 60)
print("HIGH NUMERIC CORRELATIONS")
print("=" * 60)

numeric_df = df.select_dtypes(include=np.number)

if numeric_df.shape[1] > 1:
    corr = numeric_df.corr()

    pairs = []

    for i in range(len(corr.columns)):
        for j in range(i + 1, len(corr.columns)):
            value = corr.iloc[i, j]

            if abs(value) >= 0.90:
                pairs.append({
                    "feature_1": corr.columns[i],
                    "feature_2": corr.columns[j],
                    "correlation": round(value, 3)
                })

    if pairs:
        display(
            pd.DataFrame(pairs)
            .sort_values("correlation", key=abs, ascending=False)
        )
    else:
        print("No numeric pairs with |correlation| >= 0.90.")